# Where does a quick-commerce order lose money?

**Data:** Kaggle Quick Commerce orders (~950k rows) + columns I added myself (Delivery_Cost, Discount_Amt, Profit, In_Loss, Loss_Driver, buckets).

**Important note before reading:** the original data has **no cost columns**. `Delivery_Cost` is a modelled cost, so every profit number here depends on that model.
`Profit = Order_Value - Delivery_Cost - Discount_Amt`, so it is really *contribution before product cost (COGS)*, not net margin.

**Questions I want to answer**
1. How is delivery cost built?
2. Which orders lose money, and why?
3. Do platforms / cities differ?
4. Do discounts really help profit?
5. Is fast delivery worth it?

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

path = os.environ.get("QUICK_COMM_CSV", "data/quick_comm2.csv")
df = pd.read_csv(path)
print(df.shape)
df.head()

## 1. Cleaning

In [ ]:
# drop empty export columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# this column is actually a delivery-time bucket, so rename it
df = df.rename(columns={"Distance_time_bucket": "Delivery_Time_Bucket"})

# fix city spelling
df["City"] = df["City"].str.strip().str.title().replace({"Bengluru": "Bengaluru"})
df["Company"] = df["Company"].str.strip()

print("nulls:", df.isna().sum().sum())
print("duplicate order ids:", df["Order_ID"].duplicated().sum())
print("duplicate rows:", df.duplicated().sum())
df.info()

In [ ]:
# does Profit really equal value - cost - discount? (small rounding difference is ok)
diff = (df["Order_Value"] - df["Delivery_Cost"] - df["Discount_Amt"] - df["Profit"]).abs()
print("max difference:", diff.max())

# is In_Loss the same as Profit < 0?
print("In_Loss matches Profit<0:", ((df["Profit"] < 0).astype(int) == df["In_Loss"]).all())

# delivery time outliers
q1, q3 = df["Delivery_Time_Min"].quantile([0.25, 0.75])
fence = q3 + 1.5 * (q3 - q1)
print("rows above IQR fence:", (df["Delivery_Time_Min"] > fence).sum(),
      "| max time:", df["Delivery_Time_Min"].max())

Everything reconciles. Delivery times are realistic (5-40 min), so I **kept** the outliers.

## 2. New columns
I use two margin measures:
- *per-order margin* (average of each order's %) - gets pulled down a lot by tiny baskets
- *weighted margin* = total profit / total order value - this is what a P&L would show

In [ ]:
df["Margin_pct"] = 100 * df["Profit"] / df["Order_Value"]

bins = [0, 50, 100, 150, 200, 300, 600, 1000, np.inf]
labels = ["<50", "50-99", "100-149", "150-199", "200-299", "300-599", "600-999", "1000+"]
df["Basket_Band"] = pd.cut(df["Order_Value"], bins=bins, labels=labels, right=False)

df[["Order_Value", "Delivery_Time_Min", "Distance_Km", "Delivery_Cost", "Profit"]].describe().round(2)

In [ ]:
df[["Order_Value", "Delivery_Time_Min", "Distance_Km", "Profit"]].hist(figsize=(10, 7), bins=50)
plt.tight_layout()
plt.show()

In [ ]:
# are categories balanced?
for col in ["Company", "City", "Product_Category", "Payment_Method"]:
    print((df[col].value_counts(normalize=True) * 100).round(1), "\n")

**Observation:** every platform, city, category and payment method has almost the same share of orders. Real markets are never this even, so the data is probably **simulated**. Platform/city rankings below are therefore a demo of method, not facts about real companies.

## 3. How is Delivery_Cost built?
Before reading any loss, I check whether cost is measured or just a formula.

In [ ]:
slope, intercept = np.polyfit(df["Distance_Km"], df["Delivery_Cost"], 1)
r2 = df["Distance_Km"].corr(df["Delivery_Cost"]) ** 2
print(f"Delivery_Cost = {intercept:.2f} + {slope:.2f} * km   (R2 = {r2:.4f})")

sample = df.sample(20000, random_state=1)
plt.scatter(sample["Distance_Km"], sample["Delivery_Cost"], s=3, alpha=0.3)
plt.xlabel("Distance (km)")
plt.ylabel("Delivery cost (Rs)")
plt.title("Delivery cost vs distance")
plt.show()

**Observation:** if R2 is about 1, cost is a straight-line formula of distance. So the smallest basket that avoids a loss simply rises with distance (before discounts: `intercept + slope * km`). Any basket larger than the cost of the longest trip plus its discount can never lose money in this model.

In [ ]:
# break-even basket at a few distances (ignoring discount)
for km in [2, 5, 10, 15]:
    print(f"{km} km -> break-even basket about Rs {intercept + slope * km:.0f}")

## 4. Who loses money?

In [ ]:
loss = df[df["In_Loss"] == 1]
print("loss orders:", len(loss), f"({df['In_Loss'].mean() * 100:.2f}% of all orders)")
print("total deficit: Rs", round(-loss["Profit"].sum()))
print("avg loss per loss order: Rs", round(-loss["Profit"].mean(), 2))
print("largest loss-making basket: Rs", loss["Order_Value"].max())

In [ ]:
# The Loss_Driver column
print(loss["Loss_Driver"].value_counts())

**Careful here:** earlier I thought "100% of losses are Low Value Orders" was a finding. It is not. The `Loss_Driver` label tests `Order_Value < 300` first, and no loss-making basket is above 300, so every loss gets that label automatically. It is circular.
The real statement is: **an order loses money when basket value < delivery cost + discount**, and delivery cost depends on distance.

In [ ]:
heat = df.pivot_table(index="Basket_Band", columns="Distance_Bucket",
                      values="In_Loss", aggfunc="mean", observed=True) * 100

plt.figure(figsize=(7, 4.5))
sns.heatmap(heat, annot=True, fmt=".0f", cmap="Reds")
plt.title("% of orders losing money (basket size x distance)")
plt.show()

**Observation:** losses sit in the small-basket + long-distance corner. That is the actual pattern.
(I did not rely on the correlation matrix: Profit vs Order_Value and Cost vs Distance are correlated only because I defined them that way.)

## 5. Platforms and cities

In [ ]:
by_company = df.groupby("Company").agg(
    orders=("Order_ID", "count"),
    avg_basket=("Order_Value", "mean"),
    avg_profit=("Profit", "mean"),
    avg_margin_pct=("Margin_pct", "mean"),
    total_profit=("Profit", "sum"),
    total_value=("Order_Value", "sum"),
    loss_rate_pct=("In_Loss", "mean"),
)
by_company["weighted_margin_pct"] = 100 * by_company["total_profit"] / by_company["total_value"]
by_company["loss_rate_pct"] *= 100
by_company = by_company.drop(columns=["total_profit", "total_value"]).round(2)
by_company.sort_values("avg_margin_pct")

**Note:** "loss rate" = share of orders with negative profit. It is *not* customer churn - the data has no customer id or repeat orders, so churn can't be measured.
The platform with the lowest margin also tends to have the smallest average basket, which fits the basket-size explanation.

In [ ]:
# city x company: average profit compared with that company's own average
pivot = df.pivot_table(index="City", columns="Company", values="Profit", aggfunc="mean")
gap = pivot - df.groupby("Company")["Profit"].mean()

plt.figure(figsize=(10, 5.5))
sns.heatmap(gap.round(0), annot=True, fmt=".0f", cmap="RdYlGn", center=0)
plt.title("Avg profit per order vs platform's own average (Rs)")
plt.show()

In [ ]:
# why are some cities worse? profit = basket - delivery cost - discount
city = df.groupby("City").agg(
    avg_profit=("Profit", "mean"),
    avg_basket=("Order_Value", "mean"),
    avg_delivery_cost=("Delivery_Cost", "mean"),
    avg_discount=("Discount_Amt", "mean"),
    avg_km=("Distance_Km", "mean"),
    loss_rate_pct=("In_Loss", "mean"),
)
city["loss_rate_pct"] *= 100
city = city.round(1).sort_values("avg_profit")
print(city)

best, worst = city.index[-1], city.index[0]
d = city.loc[best] - city.loc[worst]
print(f"\n{best} vs {worst}: profit gap Rs {d['avg_profit']:.1f} = "
      f"basket gap {d['avg_basket']:.1f} - cost gap {d['avg_delivery_cost']:.1f} - discount gap {d['avg_discount']:.1f}")

**Observation:** the split above shows whether the city gap comes from basket size or from trip cost (distance). The same city pattern shows up for every platform, so it looks like a *market* effect, not a platform one.

## 6. Do discounts help profit?

In [ ]:
naive = df.groupby("Discount_Applied").agg(
    orders=("Order_ID", "count"),
    avg_basket=("Order_Value", "mean"),
    avg_discount=("Discount_Amt", "mean"),
    avg_profit=("Profit", "mean"),
).round(2)
print(naive)
print("profit ratio (disc / no disc):", round(naive.loc[1, "avg_profit"] / naive.loc[0, "avg_profit"], 2))
print("basket ratio (disc / no disc):", round(naive.loc[1, "avg_basket"] / naive.loc[0, "avg_basket"], 2))

Discounted orders look much more profitable, but they also have much bigger baskets. So I compare **inside the same basket band**.

In [ ]:
band = df.pivot_table(index="Basket_Band", columns="Discount_Applied",
                      values="Profit", aggfunc="mean", observed=True).round(1)
band.columns = ["no_discount", "discount"]
band["gap"] = band["discount"] - band["no_discount"]
band

**Observation:** if the gap is small or negative inside bands, the headline "discounted orders are X% more profitable" is just a basket-size effect (people who get discounts buy more). It can't be called causal without an experiment (A/B test).

In [ ]:
# how big is the discount as a share of the basket?
d_rate = df.loc[df["Discount_Applied"] == 1, "Discount_Amt"] / df.loc[df["Discount_Applied"] == 1, "Order_Value"]
d_rate.describe(percentiles=[0.05, 0.5, 0.95]).round(4)

In [ ]:
# effect size instead of only a p-value (with ~950k rows, p is always ~0)
a = df.loc[df["Discount_Applied"] == 1, "Profit"]
b = df.loc[df["Discount_Applied"] == 0, "Profit"]
diff = a.mean() - b.mean()
se = np.sqrt(a.var() / len(a) + b.var() / len(b))
pooled_sd = np.sqrt((a.var() + b.var()) / 2)
print(f"mean profit difference: Rs {diff:.1f}  (95% CI {diff - 1.96 * se:.1f} to {diff + 1.96 * se:.1f})")
print(f"Cohen's d: {diff / pooled_sd:.2f}")

## 7. Is speed worth it?

In [ ]:
speed = df.groupby("Delivery_Time_Bucket").agg(
    orders=("Order_ID", "count"),
    avg_profit=("Profit", "mean"),
    avg_cost=("Delivery_Cost", "mean"),
    avg_km=("Distance_Km", "mean"),
    avg_rating=("Customer_Rating", "mean"),
    loss_rate_pct=("In_Loss", "mean"),
)
speed["loss_rate_pct"] *= 100
print(speed.round(2))
print("corr(time, distance):", round(df["Delivery_Time_Min"].corr(df["Distance_Km"]), 3))

In [ ]:
# hold distance fixed
df.pivot_table(index="Distance_Bucket", columns="Delivery_Time_Bucket",
               values="Profit", aggfunc="mean", observed=True).round(1)

**Observation:** fast deliveries look profitable mainly because they are short trips. If profit differences shrink inside a distance bucket, then "speed = profit" really means "near = profit". Either way this is an association, not proof that speeding up an order adds profit.

## 8. Export for SQL / BigQuery / Streamlit app

In [ ]:
os.makedirs("data", exist_ok=True)

out = df.drop(columns=["Margin_pct", "Basket_Band"])
out.columns = [c.lower() for c in out.columns]

out.to_csv("data/orders_clean.csv", index=False)
out.to_parquet("data/orders_clean.parquet", index=False)   # needs pyarrow

slim_cols = ["order_id", "company", "city", "product_category", "order_value", "distance_km",
             "delivery_time_min", "discount_applied", "delivery_cost", "discount_amt", "profit"]
out[slim_cols].to_parquet("data/orders.parquet", index=False)
print("saved")

## Summary

| Earlier idea | What I concluded |
|---|---|
| ~9.8% of orders lose money | Holds, but only under the *modelled* delivery cost |
| "100% of losses are low-value orders" | Circular - comes from how the label was written. Real cause: basket < delivery cost + discount |
| "Distance doesn't cause losses" | Wrong - cost is a formula of distance, so distance sets the break-even basket |
| "Discounts make orders 59% more profitable" | Mostly a basket-size effect; compare inside basket bands |
| "Lower attrition" | It is loss-order rate; churn can't be measured here |
| "Speed = profit" | Confounded with distance |
| Order value vs profit correlation ~1 | Mechanical, not an insight - removed |
| Platform / city rankings | Method demo only - data looks simulated |

**Limitations:** no real cost data, no dates, no customer id, probably synthetic data.
**Next step:** the Streamlit what-if app lets you change distance, basket size and discount and see where the break-even moves.